In [10]:
%%writefile src/business_entity_resolution/features.py

import pandas as pd
from rapidfuzz.fuzz import ratio


def similarity(value1, value2):

    if pd.isna(value1) or pd.isna(value2):
        return 0.0

    value1 = str(value1)
    value2 = str(value2)

    if not value1 or not value2:
        return 0.0

    return ratio(value1, value2) / 100.0


def calculate_pair_features(record1, record2):

    name_score = similarity(
        record1.get("normalized_name", ""),
        record2.get("normalized_name", "")
    )

    address_score = similarity(
        record1.get("normalized_address", ""),
        record2.get("normalized_address", "")
    )

    country_score = similarity(
        record1.get("normalized_country", ""),
        record2.get("normalized_country", "")
    )

    return {
        "name_similarity": name_score,
        "address_similarity": address_score,
        "country_similarity": country_score,
    }


def create_features(source1, source2, candidates):

    s1 = source1.set_index("entity_id")
    s2 = source2.set_index("entity_id")

    rows = []

    for _, candidate in candidates.iterrows():

        id1 = candidate["source1_id"]
        id2 = candidate["source2_id"]

        if id1 not in s1.index or id2 not in s2.index:
            continue

        features = calculate_pair_features(
            s1.loc[id1],
            s2.loc[id2]
        )

        rows.append({
            "source1_id": id1,
            "source2_id": id2,
            **features
        })

    return pd.DataFrame(rows)

Overwriting src/business_entity_resolution/features.py
